In [27]:
"""
将单倍型频率和表型信息合并，并且做成可以直接用于机器学习的训练集和测试集
CatBoost可以读入缺失值的，其他的深度学习不可以，所以要另外构建数据集
"""
import os
import numpy as np
import pandas as pd

from sklearn import preprocessing

WORK_PATH = "/share/home/zhoujb/project/riceDataComb/processData/dataFromScience/"

In [39]:
df_matrix = pd.read_table(os.path.join(WORK_PATH, "phen_matrix_1.txt"))
df_per = pd.read_table(os.path.join(WORK_PATH, "18k_150gene_hap_per.txt"))
#inte_sam_list = list(set(df_matrix["ID"].unique()).intersection(df_per["sample"].unique()))

df_merge = df_matrix.merge(df_per, left_on="ID", right_on="sample", how="left")
df_merge = df_merge.drop(columns=["sample"])

In [46]:
df_merge

,ID,region,Heading date,Plant height,Culm length,Panicle length,Panicle number,Leaf length,Leaf width,Leaf angle,...,LOC_Os11g11790,LOC_Os12g34850,LOC_Os11g34300,LOC_Os12g43550,LOC_Os11g47140,LOC_Os11g46200,LOC_Os12g29220,LOC_Os12g28750,LOC_Os11g45620,LOC_Os11g38580
0,N1-10-10A,Shanghai,103.000000,75.0,57.4,17.6,9.333333,30.1,1.9,13.0,...,0.084089,0.465501,0.093643,0.524727,0.601270,0.597090,0.456436,0.456870,0.624505,0.096737
1,N1-10-10B,Shanghai,84.333333,160.9,139.2,21.7,21.333333,37.9,1.9,16.0,...,0.084089,0.110363,0.093643,0.137723,0.154932,0.085337,0.118235,0.118126,0.085229,0.096737
2,N1-10-10C,Shanghai,94.666667,197.3,161.5,35.8,10.666667,58.2,2.3,32.0,...,0.084089,0.110363,0.093643,0.137723,0.601270,0.597090,0.118235,0.118126,0.624505,0.096737
3,N1-10-10D,Shanghai,99.666667,195.6,165.7,29.9,14.333333,45.5,1.6,27.0,...,0.511210,0.004126,0.497367,0.008794,0.154932,0.085337,0.456436,0.456870,0.085229,0.096737
4,N1-10-11A,Shanghai,82.333333,145.5,126.1,19.4,24.000000,31.9,1.4,15.0,...,0.511210,0.110363,0.093643,0.137723,0.601270,0.597090,0.118235,0.118126,0.624505,0.573096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
55258,M-9-7E,Hainan,106.000000,174.0,141.8,32.2,21.000000,22.8,1.4,30.0,...,0.511210,0.110363,0.497367,0.137723,0.601270,0.597090,0.118235,0.118126,0.624505,0.573096
55259,M-9-8E,Hainan,94.000000,128.5,102.5,26.0,31.000000,39.2,1.7,30.0,...,0.511210,0.055480,0.093643,0.137723,0.154932,0.260084,0.062754,0.063026,0.052766,0.096737
55260,M-9-9A,Hainan,96.000000,103.0,82.5,20.5,27.000000,21.9,1.3,30.0,...,0.052386,0.465501,0.056566,0.524727,0.601270,0.597090,0.118235,0.118126,0.624505,0.060800
55261,M-9-9B,Hainan,98.000000,138.5,114.5,24.0,19.000000,34.3,1.8,10.0,...,0.511210,0.110363,0.093643,0.146463,0.154932,0.260084,0.118235,0.118126,0.052766,0.096737


In [47]:
for item in ['Heading date', 'Plant height', 'Culm length', 'Panicle length', 
             'Panicle number', 'Leaf length', 'Leaf width', 'Leaf angle']:
    tmp_data = df_merge.copy().dropna(subset=item)
    # 保留 NaN比例 <=50% 的列
    tmp_data = tmp_data.loc[:, tmp_data.isna().mean() <= 0.5].copy()
    
    tmp_data = tmp_data.rename(columns={x:"fea_{}".format(x) for x in tmp_data.columns if x not in [item, 'ID', 'region']})
    tmp_data.to_csv(os.path.join(WORK_PATH, "trainTestData/data-{}.txt".format(item)), sep="\t", index=0)

    # 每个 location + year 抽样 10%
    test_df = (
        tmp_data.groupby(["region"], group_keys=False)
        .apply(lambda x: x.sample(frac=0.1, random_state=42))
    )
    # 剩余为训练集
    train_df = tmp_data.drop(test_df.index).copy()

    feat_col = [c for c in tmp_data.columns if c.startswith("fea_")]
    target_col = item

    scale_tool = preprocessing.StandardScaler()
    scale_tool.fit(train_df.loc[:, feat_col])
    train_df.loc[:, feat_col] = scale_tool.transform(train_df.loc[:, feat_col])
    test_df.loc[:, feat_col] = scale_tool.transform(test_df.loc[:, feat_col])

    train_df.to_csv(os.path.join(WORK_PATH, "trainTestData/train-{}.txt".format(item)), sep="\t", index=0)
    test_df.to_csv(os.path.join(WORK_PATH, "trainTestData/test-{}.txt".format(item)), sep="\t", index=0)
    
print("ALL DONE")

/tmp/ipykernel_158687/223534336.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_158687/223534336.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_158687/223534336.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is

ALL DONE


In [49]:
for item in ['Heading date', 'Plant height', 'Culm length', 'Panicle length', 
             'Panicle number', 'Leaf length', 'Leaf width', 'Leaf angle']:

    tmp_data = df_merge.copy().dropna(subset=[item])

    # 保留 NaN比例 <=50% 的列
    tmp_data = tmp_data.loc[:, tmp_data.isna().mean() <= 0.5].copy()

    tmp_data = tmp_data.rename(
        columns={x: f"fea_{x}" for x in tmp_data.columns if x not in [item,'ID','region']}
    )

    feature_cols = [c for c in tmp_data.columns if c.startswith("fea_")]

    # =========================
    # Step1 先划分 train/test
    # =========================

    test_df = (
        tmp_data.groupby(["region"], group_keys=False)
        .apply(lambda x: x.sample(frac=0.1, random_state=42))
    )

    train_df = tmp_data.drop(test_df.index).copy()

    # =========================
    # Step2 用 train 计算 median
    # =========================

    train_group_median = (
        train_df.groupby(["region"])[feature_cols]
        .median()
    )

    global_median = train_df[feature_cols].median()

    # =========================
    # Step3 定义 fill 函数
    # =========================

    def fill_by_group(df, group_median, global_median):

        df = df.copy()

        for loc, idx in df.groupby(["region"]).groups.items():

            if loc in group_median.index:

                med = group_median.loc[loc]

                df.loc[idx, feature_cols] = df.loc[idx, feature_cols].fillna(med)

        df[feature_cols] = df[feature_cols].fillna(global_median)

        return df

    # =========================
    # Step4 填补缺失
    # =========================

    train_df = fill_by_group(train_df, train_group_median, global_median)
    test_df  = fill_by_group(test_df,  train_group_median, global_median)

    # =========================
    # Step5 标准化（只用train）
    # =========================

    scaler = preprocessing.StandardScaler()

    scaler.fit(train_df[feature_cols])

    train_df[feature_cols] = scaler.transform(train_df[feature_cols])
    test_df[feature_cols]  = scaler.transform(test_df[feature_cols])

    # =========================
    # 保存数据
    # =========================

    train_df.to_csv(
        os.path.join(WORK_PATH,f"trainTestData/trainFill-{item}.txt"),
        sep="\t",
        index=False
    )

    test_df.to_csv(
        os.path.join(WORK_PATH,f"trainTestData/testFill-{item}.txt"),
        sep="\t",
        index=False
    )

print("ALL DONE")

/tmp/ipykernel_158687/689688476.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_158687/689688476.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(frac=0.1, random_state=42))
/tmp/ipykernel_158687/689688476.py:21: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is

ALL DONE
